# 02 — Model detective (Student workbook)

**Time:** 70 minutes
**Question:** when does the dataset contain enough information to distinguish
a sphere from an ellipsoid?

The file name deliberately reveals nothing about the generating model. Split
the work: one partner advocates for a sphere, the other for an ellipsoid.

> **Lecture connection**
>
> - Slide 22, *Welcome to the SANS zoo*: many shapes share a plateau and a
>   decay and differ in oscillation depth and mid-q slope. Browse the
>   [SasView model marketplace](https://marketplace.sasview.org/) for the full zoo.
> - Slide 23, *Name that shape!*: panels B and C "look alike over a narrow
>   Q-window", and an extended Q-range shows the difference. This notebook is
>   that statement made quantitative.
> - Slide 19, *Guinier analysis*: the $R_g$ table. Every shape looks the same at
>   low Q apart from $I(0)$ and $R_g$.
> - Slide 33, *Different models, similar fits*: bring your figures from this
>   notebook to the plenary discussion.

In [ ]:
from pathlib import Path
import sys

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
sys.path.insert(0, str(ROOT / 'src'))

import matplotlib.pyplot as plt
import numpy as np
import sans_fitter
from sans_fitter import SANSFitter

DATA = ROOT / 'data'
print(f'Using sans_fitter from {Path(sans_fitter.__file__).resolve()}')
%matplotlib inline


In [ ]:
filename = DATA / 'mystery_shape.csv'
values = np.genfromtxt(filename, delimiter=',', names=True)
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(values['Q'], values['I'], yerr=values['dI'], fmt='o', ms=3)
ax.set(xscale='log', yscale='log', xlabel=r'$q$ / Å$^{-1}$', ylabel=r'$I(q)$')
ax.grid(alpha=0.25)
plt.show()


### Look before you fit

The lecture's fitting strategy (slide 30) starts with reading the data, not
with the optimizer. Compare this curve with `sphere_reference.csv` from
notebook 01:

- Are the form-factor minima as deep and sharp? Shallow, washed-out minima
  can mean polydispersity or smearing, but also **anisotropy**: an
  orientationally averaged ellipsoid has a different dimension in every
  direction, so its minima fall at different q and blur together.
- Where does the plateau bend over, and what slope follows it (slide 20)?
  A very long object would show an extended $Q^{-1}$ region. A moderately
  elongated one shows only a hint of it.
- Notice that these data are noisier than in notebook 01. How might that
  affect your ability to tell two models apart?

## Exercise 1 — Competing explanations

You now know the full workflow, so this notebook gives less code. Implement a
reusable sphere fit first. Keep SLD values fixed and vary only radius, scale,
and background. Compare residual patterns, not just the numerical objective.

In [ ]:
def fit_sphere(qmin=None, qmax=None):
    # Fit a sphere, optionally over only part of the measured q range.
    fitter = SANSFitter()
    fitter.load_data(str(filename))
    if qmin is not None or qmax is not None:
        fitter.set_q_range(qmin=qmin, qmax=qmax)

    # YOUR CODE HERE
    # Select the model, configure its parameters, and return the fitter and
    # result. Choose your own defensible initial values and bounds.
    raise NotImplementedError('Implement the sphere fitting workflow')


sphere, sphere_result = fit_sphere()
sphere.plot_results(show_residuals=True, log_scale=True)


### A competing model

Create a **new** `SANSFitter` for the ellipsoid so the two fits remain
independent. After `set_model('ellipsoid')`, use `get_params()` to discover the
model's parameter names. Decide whether each should be fixed or varied.

In [ ]:
ellipsoid = SANSFitter()
ellipsoid.load_data(str(filename))

# Select 'ellipsoid', call `get_params()` to discover its two radius
# parameters, then configure and fit it.

# YOUR CODE HERE
raise NotImplementedError('Configure and fit the ellipsoid model')


In [ ]:
print('Sphere objective:', sphere_result['chisq'])
print('Ellipsoid objective:', ellipsoid_result['chisq'])
for name in ['radius_polar', 'radius_equatorial']:
    print(name, ellipsoid_result['parameters'][name]['formatted'])


## Exercise 2 — Remove information

Repeat both fits using only `0.004 ≤ q ≤ 0.025 Å⁻¹`. Predict what will happen
before running the cells. Which shape features have disappeared from view?

*Before you run:* convert the window to real space with $d \approx 2\pi/Q$,
as in notebook 00. What is the smallest distance still resolved, and how does
it compare with the particle dimensions you just fitted?

In [ ]:
# Reuse `fit_sphere` with qmin=0.004 and qmax=0.025.
# Then build an ellipsoid fit with the same q limits.

# YOUR CODE HERE
raise NotImplementedError('Fit both models over the restricted q range')


### Why the models become indistinguishable

The lecture called the Guinier law *universal*: at low Q every shape gives
$I(Q) \approx I(0)\exp(-Q^2R_g^2/3)$, which is valid for $QR_g \lesssim 1.3$
(slide 19). In that region the data carry essentially **two numbers**, $I(0)$
and $R_g$. Any model that can reproduce those two numbers fits equally well.

Test this with your own fitted parameters:

1. For a spheroid with equatorial radius $R_e$ and polar radius $R_p$,
   $R_g^2 = (2R_e^2 + R_p^2)/5$. For a sphere, $R_g^2 = \tfrac{3}{5}R^2$.
   Compute both from your **restricted-range** fits. How close are they?
2. Compute $Q_{\max}R_g$ for the restricted window. Where does it sit relative
   to the Guinier limit of 1.3?
3. Using the full-range ellipsoid fit, what radius would a sphere *with the
   same* $R_g$ have? Is this close to the radius your restricted sphere fit
   found?

This is **non-identifiability**: the data are compatible with a family of
parameter sets, so the optimizer's single answer is not a measurement of
shape. Notebook 04 shows the same idea as correlated posterior distributions.

## Verdict

Write a short referee-style verdict. Include one reason based on residuals, one
based on parameter plausibility, and one follow-up measurement that would make
the distinction stronger.

Follow-up measurements from the lecture to consider:

- **Extend q.** A shorter detector distance or a shorter wavelength raises
  $Q_{\max}$ (slides 8 and 37–39). Ask which q range the shape information
  lives in.
- **Change the contrast.** H/D variation (slides 13 and 15) changes $\Delta\rho$
  and so constrains the scale–contrast degeneracy, but does contrast alone
  change the *shape* of the curve for a homogeneous particle?
- **Complementary techniques.** Slide 6 asked when SANS and TEM would disagree
  about a size. Here, what could a micrograph settle that SANS cannot, and
  what could it get wrong?
- **Field or flow alignment.** Anisotropic objects give anisotropic 2D
  patterns when aligned (compare slide 27): a radially averaged 1D curve has
  thrown that information away.

In [ ]:
# TODO: defend a model using residuals and physical plausibility.
conclusion = 'Write your evidence-based conclusion here.'
print(conclusion)


## Further reading

- [SasView `ellipsoid` model](https://www.sasview.org/docs/user/models/ellipsoid.html): parameter
  definitions. `radius_polar` lies along the rotation axis, so $R_p > R_e$ is
  prolate (cigar-like) and $R_p < R_e$ is oblate (disc-like).
- [Jacques & Trewhella, *Small-angle scattering for structural biology: expanding the frontier while avoiding the pitfalls*, Protein Sci. 19, 642 (2010)](https://pmc.ncbi.nlm.nih.gov/articles/PMC2867006/): the discussion of model ambiguity and of reporting what the data can
  and cannot support.